<a href="https://colab.research.google.com/github/google-ai-edge/litert-samples/blob/main/benchmark/developer_device_platform/embedding_gemma_ddp_benchmark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##### Copyright 2026 The AI Edge Authors.

In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.
# ==============================================================================

# Benchmarking EmbeddingGemma on DDP with the `device-run` CLI

This notebook benchmarks an **EmbeddingGemma** model with [LiteRT-LM](https://github.com/google-ai-edge/LiteRT-LM)'s [`embedding_litert_lm_main`](https://github.com/google-ai-edge/LiteRT-LM/blob/main/runtime/engine/embedding_litert_lm_main.cc) on physical Android phones, using [Developer Device Platform](https://docs.cloud.google.com/developer-device-platform/overview) (DDP) to allocate the hardware.

It calls the DDP CLI, `gcloud beta device-run`, directly. That gives you:

* **Any runtime.** DDP runs whatever Android binary you hand it. Here it is LiteRT-LM's embedding binary; benchmarking another runtime means changing the executable and its flags.
* **Lower-level control.** You choose every flag, and you get back every file the benchmark writes, along with each device's `logcat.txt`.
* **Many devices at once.** One command fans the same run out across a list of devices.

A wrapper script runs four configurations on each device, a short text prompt and a text + image + audio (tri-modal) input, each on **CPU and GPU**:

* `text_cpu`, `text_gpu`: text only.
* `tri_modal_cpu`, `tri_modal_gpu`: text, an image and an audio clip embedded together.

> Benchmarking an LLM instead? See [`ddp_cli_benchmark.ipynb`](ddp_cli_benchmark.ipynb).

## 🛠️ 1. Environment setup

In [ ]:
# @title Set up your GCP project for DDP
# @markdown Set your GCP project id (DDP sessions are billed to it). This cell
# @markdown logs in to gcloud and enables the Device Run API.

ddp_gcp_project = "your-own-gcp-project-id" # @param {type:"string"}

# @markdown The GCS bucket DDP uses for inputs and results. Leave blank to use
# @markdown the default `{project_id}-devicerun`.
custom_bucket_id = "" # @param {type:"string"}

bucket_id = custom_bucket_id if custom_bucket_id else f"{ddp_gcp_project}-devicerun"

if not ddp_gcp_project or ddp_gcp_project == "your-own-gcp-project-id":
  raise ValueError("Please specify a valid GCP Project ID in the form above.")

# Login to gcloud and set Application Default Credentials via Colab Auth
from google.colab import auth
auth.authenticate_user()

# Enable the devicerun API
!gcloud services enable devicerun.googleapis.com --project {ddp_gcp_project}

# DDP stages inputs and writes results here; create it if it is missing.
!gcloud storage buckets describe "gs://{bucket_id}" >/dev/null 2>&1 \
  || gcloud storage buckets create "gs://{bucket_id}" --project={ddp_gcp_project}

print(f"Project: {ddp_gcp_project}\nBucket:  gs://{bucket_id}")

In [ ]:
# @title Download latest version of GCloud CLI
# @markdown This ensures the latest features from DDP CLI are available in the colab.

import os

# 1. Download and extract the absolute latest gcloud release directly to /opt
!wget -qO- https://dl.google.com/dl/cloudsdk/channels/rapid/downloads/google-cloud-cli-linux-x86_64.tar.gz | tar xz -C /opt

# 2. Run the silent install script
!/opt/google-cloud-sdk/install.sh -q

# 3. Add the new installation to the front of the Python environment's PATH
os.environ["PATH"] = f"/opt/google-cloud-sdk/bin:{os.environ['PATH']}"

# 4. Explicitly install the beta components using the newly downloaded CLI
!gcloud components install beta --quiet

## 🧠 2. Download the EGv2 model

This notebook benchmarks [EmbeddingGemma 2 740M](https://huggingface.co/litert-community/embeddinggemma-2-740m-litert-lm), the EmbeddingGemma 2 variant that embeds text, images and audio, which the tri-modal runs need.

The cell below downloads its CPU/GPU file, `embeddinggemma-2-740m.litertlm` (about 485 MB), from Hugging Face. DDP pushes files to devices from GCS, so `gcloud` uploads the model to your bucket when the session is submitted.

In [ ]:
# @title Download the model
# @markdown Downloads EmbeddingGemma 2 740M from Hugging Face.
import os
import subprocess

MODEL_URL = ("https://huggingface.co/litert-community/embeddinggemma-2-740m-litert-lm"
             "/resolve/main/embeddinggemma-2-740m.litertlm")
MODEL_PATH = os.path.abspath("embeddinggemma-2-740m.litertlm")

# Download to a temporary name so an interrupted download is never reused.
if not os.path.exists(MODEL_PATH):
  subprocess.run(["curl", "-sSfL", MODEL_URL, "-o", MODEL_PATH + ".part"],
                 check=True)
  os.rename(MODEL_PATH + ".part", MODEL_PATH)

print(f"Model: {MODEL_PATH} ({os.path.getsize(MODEL_PATH) / 1e6:,.0f} MB)")

## 📱 3. Choose devices

DDP refers to devices by their device ID, such as `pa3q-35` or `m2q-36`. Browse the [device catalog](https://docs.cloud.google.com/developer-device-platform/device-catalog) to see what is available, or list it from a scratch cell:

```bash
!gcloud beta device-run devices list --project="{ddp_gcp_project}" --location="global"
```

Enter the IDs, comma-separated, in the cell below. DDP runs one job per device.

In [ ]:
# @title Set the target devices
# @markdown Comma-separated DDP device IDs:
ddp_target_devices = "pa3q-35,m2q-36" # @param {type: "string"}

devices = [d.strip() for d in ddp_target_devices.split(",") if d.strip()]
if not devices:
  raise ValueError("Enter at least one device ID.")
ddp_target_devices = ",".join(devices)
print(f"{len(devices)} device(s): {ddp_target_devices}")

## 📦 4. Benchmark binaries

The binaries are hosted in the public LiteRT release bucket on Google Cloud Storage under `gs://litert/binaries/latest/android_arm64/litert_lm/`. The embedding benchmark needs two files from it:

| File | Purpose |
| :--- | :--- |
| `embedding_litert_lm_main` | The embedding benchmark executable. |
| `libLiteRtOpenClAccelerator.so` | **GPU only.** The OpenCL delegate, loaded at runtime when `--backend=gpu` is passed. |

Because the OpenCL delegate is opened at *runtime* (`dlopen`) rather than linked at build time, the Android loader has to be told where to find it. This is why the wrapper script sets `LD_LIBRARY_PATH=/data/local/tmp`.

> **Note:** The tri-modal runs need an `embedding_litert_lm_main` recent enough to preprocess images for this model. An older build fails them with `Image preprocessor is not available for raw image input`, while the text runs still pass.

In [ ]:
# @title Set the binaries
# @markdown `embedding_litert_lm_main` and `libLiteRtOpenClAccelerator.so` from
# @markdown the public LiteRT release.
import subprocess

BIN_DIR = "gs://litert/binaries/latest/android_arm64/litert_lm"
EMBED_BIN = f"{BIN_DIR}/embedding_litert_lm_main"
OPENCL_LIB = f"{BIN_DIR}/libLiteRtOpenClAccelerator.so"

WORK = "/data/local/tmp"  # where DDP pushes files on the device


def gcs_exists(uri: str) -> bool:
  return subprocess.run(["gcloud", "storage", "ls", uri],
                        capture_output=True).returncode == 0


# Fail now, not after DDP has allocated the devices.
for uri in [EMBED_BIN, OPENCL_LIB]:
  if not gcs_exists(uri):
    raise FileNotFoundError(f"Not found, or not readable by you: {uri}")

print(f"Binary: {EMBED_BIN}")

### 📝 4.1 Wrapper script

DDP is handed a shell script, `run_all.sh`, instead of the binary, and the script runs the binary once per configuration. The cells below write the script, then upload it to GCS with the test image and audio clip it uses. That buys a few things a single invocation cannot:

* **Several configurations per device.** Text and tri-modal inputs, each on CPU and GPU.
* **Warm-start measurement.** Each configuration runs *twice*. The first pass is discarded; it only writes the XNNPACK / ML Drift caches next to the model. The second pass is the measured one.
* **Thermal cooldown.** A `sleep 20` between configurations stops one run from throttling the next.
* **Device identification.** The script records `getprop ro.product.model`, so results carry the real model name. DDP itself only labels results `job-000`, `job-001`, ….

Flags worth knowing:

* **`--use_mmap=true`** memory-maps the model weights instead of reading them into RAM, which lowers the peak resident memory.
* **`--num_iterations=50`** runs 50 measured embeddings per configuration (after the binary's own warm-up iterations) and reports their average latency.
* **`--num_cpu_threads=4`** is set for CPU runs only. On GPU the driver manages its own threads.
* **`--visual_token_budget=70`** caps the image at 70 tokens so that, with the audio clip and the short prompt, the tri-modal input fits in the `--max_input_length=128` context.

#### Where the results come from
In `--benchmark` mode, `embedding_litert_lm_main` prints its results (average latency, peak memory, and per-encoder times such as `vision_executor` and `audio_executor`) to the console. The script saves each configuration's output to `/data/local/tmp/out/<config>.log`, next to its exit code, and `--paths-to-pull` brings that folder back. The results table is parsed from those files.

For GPU runs the same file is also checked for `LITERT_CL`, the OpenCL delegate's marker, to confirm the GPU was really used rather than a silent fallback to CPU.

In [ ]:
%%writefile run_all.sh
#!/system/bin/sh

WORK="/data/local/tmp"
# The OpenCL delegate is loaded at runtime, so it has to be on the loader path.
export LD_LIBRARY_PATH="$WORK"
chmod 755 "$WORK/embedding_litert_lm_main"
rm -rf "$WORK/out" && mkdir -p "$WORK/out"

# DDP reports only job-NNN. Without this the report cannot name the hardware.
getprop ro.product.model > "$WORK/out/_device.txt"

# Write the prompt to a file so it stays a single shell word.
printf '%s' "The quick brown fox jumps over the lazy dog" > "$WORK/prompt.txt"

COMMON="--model_path=$WORK/model.litertlm \
    --benchmark=true \
    --use_mmap=true \
    --report_peak_memory_footprint=true \
    --num_iterations=50 \
    --max_input_length=128 \
    --input_prompt_file=$WORK/prompt.txt"

TRI_MODAL="--image_path=$WORK/test_image_70.png \
    --visual_token_budget=70 \
    --audio_path=$WORK/test_audio_5s.wav"

# run <config> <extra flags...>
#   First pass warms the caches and is discarded; second pass is measured.
run() {
  config="$1"; shift
  echo "### $config"
  "$WORK/embedding_litert_lm_main" $COMMON "$@" >/dev/null 2>&1
  "$WORK/embedding_litert_lm_main" $COMMON "$@" >"$WORK/out/$config.log" 2>&1
  echo $? > "$WORK/out/$config.exit"
  sleep 20   # let the SoC cool so the next config is not throttled
}

run text_cpu      --backend=cpu --num_cpu_threads=4
run tri_modal_cpu $TRI_MODAL --backend=cpu --num_cpu_threads=4
run text_gpu      --backend=gpu
run tri_modal_gpu $TRI_MODAL --backend=gpu

echo "ALL DONE"

In [ ]:
# @title Upload the script and test inputs to GCS
# @markdown Uploads `run_all.sh` plus the test image and audio clip for the
# @markdown tri-modal runs, from the LiteRT-LM repo.
import uuid

# A unique path per run: overwriting a shared script would corrupt any session
# that has not started yet, because DDP fetches the script when the job starts.
run_tag = uuid.uuid4().hex[:8]
INPUTS = f"gs://{bucket_id}/scripts/{run_tag}"

TESTDATA = "https://raw.githubusercontent.com/google-ai-edge/LiteRT-LM/main/runtime/testdata"
!curl -sSfL {TESTDATA}/test_image_70.png -o test_image_70.png
!curl -sSfL {TESTDATA}/test_audio_5s.wav -o test_audio_5s.wav
!gcloud storage cp run_all.sh test_image_70.png test_audio_5s.wav "{INPUTS}/"

print(f"Uploaded to {INPUTS}/")

## 🚀 5. Submit the benchmark

The next cell submits one session with the `android-executable` workflow, using everything staged above:

```bash
gcloud beta device-run sessions submit android-executable \
  --device="pa3q-35,m2q-36" \
  --executable=gs://.../run_all.sh \
  --other-files-to-push="...=/data/local/tmp/model.litertlm,..." \
  --paths-to-pull="/data/local/tmp/out" \
  --async
```

* **`--device`**: a comma-separated list of device IDs. DDP runs one job per device.
* **`--executable`**: the script DDP downloads to each device and runs.
* **`--other-files-to-push`**: the binary, the OpenCL delegate, the model and the test image and audio, each copied to `/data/local/tmp/` before the script starts. Local files, such as the model, are first uploaded to your bucket.
* **`--paths-to-pull`**: the script's `out` folder, copied back to your bucket when the job ends. DDP also uploads each device's `logcat.txt`.
* **`--async`**: return as soon as the session is created. Four configurations take several minutes per device, so the cell after it polls the session and downloads the results when it finishes.

In [ ]:
# @title Submit the benchmark
# @markdown Submits one job per device, each running all four configurations.
# @markdown Uses `--async` so the next cell can report progress.
import re

push = [
    f"{MODEL_PATH}={WORK}/model.litertlm",
    f"{EMBED_BIN}={WORK}/embedding_litert_lm_main",
    f"{OPENCL_LIB}={WORK}/libLiteRtOpenClAccelerator.so",
    f"{INPUTS}/test_image_70.png={WORK}/test_image_70.png",
    f"{INPUTS}/test_audio_5s.wav={WORK}/test_audio_5s.wav",
]

submit_output = !gcloud beta device-run sessions submit android-executable \
  --project="{ddp_gcp_project}" \
  --location="global" \
  --bucket-name="{bucket_id}" \
  --device="{ddp_target_devices}" \
  --executable="{INPUTS}/run_all.sh" \
  --other-files-to-push="{','.join(push)}" \
  --paths-to-pull="{WORK}/out" \
  --executable-timeout=30m \
  --async 2>&1

print("\n".join(submit_output))

# The session ID is how every later cell finds this run.
m = re.search(r"Creating session \[([^\]]+)\]", "\n".join(submit_output))
if not m:
    raise SystemExit("Could not find a session ID; check the output above.")
session_id = m.group(1)

In [ ]:
# @title Poll until the session completes
# @markdown `sessions wait` blocks silently, so we poll `sessions describe`
# @markdown instead and print progress as jobs finish.
# @markdown
# @markdown ---
# @markdown **Polling interval (seconds):**
import subprocess
import time

POLL_SECONDS = 60 # @param {type:"integer"}

print(f"Waiting for {session_id}, polling every {POLL_SECONDS}s...\n")

start = time.time()
while True:
    out = subprocess.run(
        ["gcloud", "beta", "device-run", "sessions", "describe", session_id,
         f"--project={ddp_gcp_project}", "--location=global"],
        capture_output=True, text=True)
    # `describe` writes its status prose to stderr and the job table to stdout.
    status = (out.stderr + out.stdout).strip()

    # Keep just the status and the job tally; drop the long bucket URL.
    keep = [l for l in status.splitlines()
            if l.startswith("Session [") or l.startswith("Job status:")]
    print(f"[{(time.time() - start) / 60:5.1f} min] " + " | ".join(keep))
    if out.returncode != 0:
        print(f"  (describe exited {out.returncode})")

    if "finished with result" in status:
        break
    time.sleep(POLL_SECONDS)

## 📊 6. Read results

The first cell below (collapsed) defines the helpers that download a session's results and parse the `.log` file each configuration wrote. The dashboard then shows one row per device and configuration:

* **`latency_ms`**: average time to compute one embedding.
* **`peak_ram_mb`**: peak memory footprint of the process.
* **`vision_exec_ms`, `audio_exec_ms`**: time spent in the image and audio encoders (tri-modal runs only).
* **`capacity`**: the text encoder signature that ran, in tokens.
* **`gpu_ok`**: whether the OpenCL delegate really loaded (GPU runs only).

Any failed run, or GPU run that fell back to CPU, is listed under the table.

In [ ]:
# @title Helpers: parse the benchmark results
# @markdown Downloads a session's results and parses the console output that
# @markdown `run_all.sh` saved for each configuration.
import re
from pathlib import Path

import pandas as pd

# Session artifacts are downloaded here, and every reader below looks here.
RESULTS_DIR = "/content/ddp_results"

# Lines embedding_litert_lm_main prints in --benchmark mode.
METRIC_PATTERNS = {
    "latency_ms": r"Average Latency:\s*([\d.]+)",
    "peak_ram_mb": r"Peak system ram usage:\s*([\d.]+)",
    "vision_exec_ms": r"- vision_executor:\s*([\d.]+)",
    "audio_exec_ms": r"- audio_executor:\s*([\d.]+)",
}
METRIC_KEYS = [*METRIC_PATTERNS, "capacity"]


def read_metrics(log_path) -> dict:
    """Parses one configuration's console output.

    A run that never started leaves no log, so a missing file yields empty values.
    """
    if log_path is None or not Path(log_path).exists():
        return dict.fromkeys(METRIC_KEYS)
    body = Path(log_path).read_text(errors="replace")
    row = {}
    for key, pattern in METRIC_PATTERNS.items():
        m = re.search(pattern, body)
        row[key] = float(m.group(1)) if m else None
    # The text encoder that ran, e.g. encoder_1x128 -> a 128-token signature.
    cap = re.search(r"encoder_1x(\d+)", body)
    row["capacity"] = int(cap.group(1)) if cap else None
    return row


def gpu_delegated(log_path) -> bool:
    """True if the OpenCL delegate loaded.

    LITERT_CL is the delegate's marker; "GPU OpenCL" appears on CPU runs too.
    """
    return (log_path is not None and Path(log_path).exists()
            and "LITERT_CL" in Path(log_path).read_text(errors="replace"))


def download_session_results(session_id):
    """Copies a session's artifacts from GCS into RESULTS_DIR.

    Assumes `bucket_id` is defined as a global variable in the notebook.
    """
    !mkdir -p {RESULTS_DIR}
    !gcloud storage cp -r gs://{bucket_id}/automation/sessions/{session_id}/ {RESULTS_DIR}/


def generate_dashboard(session_id: str) -> pd.DataFrame:
    """Tabulates a session and plots latency and peak RAM for each device."""
    local_dir = Path(f"{RESULTS_DIR}/{session_id}")
    rows = []

    # The script writes <config>.exit for every configuration, even failed
    # ones, so iterating over those means a failed run still gets a row.
    for exit_file in sorted(local_dir.rglob("out/*.exit")):
        out, config = exit_file.parent, exit_file.stem  # e.g. 'text_gpu'
        dev_file = out / "_device.txt"
        device = (dev_file.read_text().strip() if dev_file.exists()
                  else re.search(r"job-\d+", str(out)).group())
        log = out / f"{config}.log"
        rows.append({
            "device": device,
            "config": config,
            "exit": exit_file.read_text().strip(),
            **read_metrics(log),
            "gpu_ok": gpu_delegated(log) if config.endswith("_gpu") else None,
        })

    if not rows:
        print(f"No results found under {local_dir} -- did the download succeed?")
        return None

    df = pd.DataFrame(rows).set_index(["device", "config"]).sort_index()
    print(df.to_string(float_format=lambda v: f"{v:,.1f}"))

    # Alert on failures or missing GPU delegation.
    bad = df[(df.exit != "0") | (df.gpu_ok == False)]  # noqa: E712
    if not bad.empty:
        print("\nFAILED or no GPU delegation: "
              + ", ".join(f"{d}/{c}" for d, c in bad.index))

    for metric in ["latency_ms", "peak_ram_mb"]:
        df[metric].unstack("device").plot.bar(title=metric, figsize=(8, 3.5), rot=0)

    return df

In [ ]:
# @title 📊 Results dashboard
# @markdown Downloads each device's results, tabulates every configuration and
# @markdown plots latency and peak RAM per device.
download_session_results(session_id)
df = generate_dashboard(session_id)